# 05 - Sticker-Linker Metrics

This notebook computes biophysical metrics relevant to IDP phase separation behavior.

## Contents
1. Compute sticker fraction and linker statistics
2. Calculate ΔG proxy from interaction maps
3. Analyze sequence metrics (FCR, NCPR, SCD)
4. Compute sticker spacing metrics (κ, Ω)
5. Generate comprehensive metrics table

In [ ]:
# Add src to path
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
# Import modules
from src.sequences import VARIANTS, get_variant
from src.intermaps import (
    compute_all_intermaps,
    InterMapConfig,
    compute_interaction_profile,
)
from src.segmentation import (
    identify_stickers_hybrid,
    create_sticker_mask,
    compute_linker_statistics,
)
from src.metrics import (
    compute_sticker_fraction,
    compute_linker_length_kappa,
    compute_mean_linker_length,
    compute_total_interaction_energy,
    compute_mean_interaction_energy,
    compute_delta_G_proxy,
    compute_sticker_interaction_strength,
    compute_FCR,
    compute_NCPR,
    compute_SCD,
    compute_kappa_charge,
    compute_aromatic_fraction,
    compute_tyrosine_fraction,
    compute_omega,
    compute_sticker_clustering,
    compute_all_metrics,
    VariantMetrics,
)
from src.plotting import (
    plot_metrics_bar,
    plot_metrics_multi_panel,
    set_publication_style,
)

In [ ]:
set_publication_style()

In [ ]:
# Load data
config = InterMapConfig(smooth=True, sigma=2.0, normalize=False)
intermaps = compute_all_intermaps(VARIANTS, config=config)

# Compute sticker masks
sticker_masks = {}
for name, variant in VARIANTS.items():
    profile = compute_interaction_profile(intermaps[name])
    sticker_bool = identify_stickers_hybrid(profile, variant, energy_threshold=-0.15)
    sticker_masks[name] = create_sticker_mask(sticker_bool)

print(f"Loaded {len(VARIANTS)} variants with interaction maps and sticker masks")

## 1. Sticker Fraction and Linker Statistics

The sticker fraction (f_sticker) is a key determinant of phase separation propensity. Higher sticker fraction generally correlates with lower critical concentration for phase separation.

In [ ]:
# Compute sticker metrics
print("Sticker-Linker Metrics:")
print("=" * 70)
print(f"{'Variant':12s} {'f_sticker':>10s} {'n_sticker':>10s} {'<L>':>10s} {'κ_linker':>10s}")
print("-" * 70)

for name, mask in sticker_masks.items():
    f_sticker = compute_sticker_fraction(mask)
    mean_L = compute_mean_linker_length(mask)
    kappa = compute_linker_length_kappa(mask)
    
    kappa_str = f"{kappa:.3f}" if not np.isnan(kappa) else "N/A"
    print(f"{name:12s} {f_sticker:10.3f} {mask.n_stickers:10d} {mean_L:10.1f} {kappa_str:>10s}")

In [ ]:
# Visualize sticker fraction
sticker_fractions = {name: compute_sticker_fraction(mask) for name, mask in sticker_masks.items()}

fig = plot_metrics_bar(
    sticker_fractions,
    title='Sticker Fraction by Variant',
    ylabel='Sticker Fraction',
    figsize=(10, 5)
)
plt.show()

In [ ]:
# Linker length distributions
from src.segmentation import compute_linker_lengths

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, name in zip(axes, ['WT', 'AllY_to_S', 'AllY_to_F']):
    mask = sticker_masks[name]
    linker_lengths = compute_linker_lengths(mask)
    
    if len(linker_lengths) > 0:
        ax.hist(linker_lengths, bins=range(0, max(linker_lengths)+2), 
                color='#3498DB', edgecolor='black', alpha=0.7)
        ax.axvline(x=np.mean(linker_lengths), color='red', linestyle='--', 
                   label=f'Mean={np.mean(linker_lengths):.1f}')
    
    ax.set_xlabel('Linker Length (residues)')
    ax.set_ylabel('Count')
    ax.set_title(f'{name}')
    ax.legend()

plt.suptitle('Linker Length Distributions', fontsize=14)
plt.tight_layout()
plt.show()

## 2. ΔG Proxy from Interaction Maps

The ΔG proxy estimates the enthalpic driving force for phase separation based on the interaction map.

In [ ]:
# Compute energy metrics
print("Energy-Based Metrics:")
print("=" * 80)
print(f"{'Variant':12s} {'Total E':>12s} {'Mean E':>12s} {'ΔG proxy':>12s} {'E_ss':>12s}")
print("-" * 80)

for name in VARIANTS:
    imap = intermaps[name]
    mask = sticker_masks[name].mask
    
    total_E = compute_total_interaction_energy(imap)
    mean_E = compute_mean_interaction_energy(imap)
    delta_G = compute_delta_G_proxy(imap, mask)
    E_ss = compute_sticker_interaction_strength(imap, mask)
    
    print(f"{name:12s} {total_E:12.2f} {mean_E:12.4f} {delta_G:12.4f} {E_ss:12.4f}")

In [ ]:
# ΔG proxy comparison
delta_G_values = {}
for name in VARIANTS:
    delta_G_values[name] = compute_delta_G_proxy(intermaps[name], sticker_masks[name].mask)

fig, ax = plt.subplots(figsize=(10, 5))

names = list(delta_G_values.keys())
values = list(delta_G_values.values())
x = np.arange(len(names))

colors = ['#F39C12' if n == 'WT' else '#3498DB' for n in names]
ax.bar(x, values, color=colors, edgecolor='black')

ax.set_xticks(x)
ax.set_xticklabels(names, rotation=45, ha='right')
ax.set_ylabel('ΔG Proxy (kT)')
ax.set_title('ΔG Proxy: Enthalpic Driving Force for Phase Separation\n(More negative = stronger driving force)')
ax.axhline(y=0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

## 3. Sequence-Based Metrics

Classical IDP metrics from sequence alone.

In [ ]:
# Compute sequence metrics
print("Sequence-Based Metrics:")
print("=" * 80)
print(f"{'Variant':12s} {'f_Y':>8s} {'f_arom':>8s} {'FCR':>8s} {'NCPR':>8s} {'SCD':>10s} {'κ_charge':>10s}")
print("-" * 80)

for name, variant in VARIANTS.items():
    f_Y = compute_tyrosine_fraction(variant)
    f_arom = compute_aromatic_fraction(variant)
    fcr = compute_FCR(variant)
    ncpr = compute_NCPR(variant)
    scd = compute_SCD(variant)
    kappa = compute_kappa_charge(variant)
    
    print(f"{name:12s} {f_Y:8.3f} {f_arom:8.3f} {fcr:8.3f} {ncpr:+8.3f} {scd:10.2f} {kappa:10.3f}")

In [ ]:
# Visualize aromatic content
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

names = list(VARIANTS.keys())
x = np.arange(len(names))

# Tyrosine fraction
f_Y = [compute_tyrosine_fraction(VARIANTS[n]) for n in names]
axes[0].bar(x, f_Y, color='#F39C12', edgecolor='black')
axes[0].set_xticks(x)
axes[0].set_xticklabels(names, rotation=45, ha='right')
axes[0].set_ylabel('Fraction')
axes[0].set_title('Tyrosine Fraction')

# Aromatic fraction
f_arom = [compute_aromatic_fraction(VARIANTS[n]) for n in names]
axes[1].bar(x, f_arom, color='#9B59B6', edgecolor='black')
axes[1].set_xticks(x)
axes[1].set_xticklabels(names, rotation=45, ha='right')
axes[1].set_ylabel('Fraction')
axes[1].set_title('Aromatic Fraction (Y+F+W)')

plt.tight_layout()
plt.show()

## 4. Sticker Spacing Metrics

The spacing and clustering of stickers affects the effective valency of interactions.

In [ ]:
# Compute spacing metrics
print("Sticker Spacing Metrics:")
print("=" * 60)
print(f"{'Variant':12s} {'Ω (spacing)':>12s} {'Clustering':>12s}")
print("-" * 60)

for name, mask in sticker_masks.items():
    omega = compute_omega(mask)
    clustering = compute_sticker_clustering(mask, distance_threshold=3)
    
    omega_str = f"{omega:.3f}" if not np.isnan(omega) else "N/A"
    print(f"{name:12s} {omega_str:>12s} {clustering:12.3f}")

In [ ]:
# Visualize sticker positions
fig, axes = plt.subplots(3, 1, figsize=(14, 6), sharex=True)

variants_to_show = ['WT', 'AllY_to_S', 'AllY_to_F']

for ax, name in zip(axes, variants_to_show):
    mask = sticker_masks[name]
    positions = mask.positions
    n = len(mask.mask)
    
    # Plot sticker positions
    ax.scatter(positions + 1, [0.5] * len(positions), c='#E74C3C', s=100, marker='|', linewidths=2)
    
    omega = compute_omega(mask)
    omega_str = f"Ω={omega:.2f}" if not np.isnan(omega) else "Ω=N/A"
    
    ax.set_xlim(0, n)
    ax.set_ylim(0, 1)
    ax.set_yticks([])
    ax.set_ylabel(name, rotation=0, ha='right', va='center')
    ax.set_title(f'{name}: {mask.n_stickers} stickers, {omega_str}', loc='left', fontsize=10)

axes[-1].set_xlabel('Residue Position')
plt.suptitle('Sticker Positions Along Sequence', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Comprehensive Metrics Table

In [ ]:
# Compute all metrics for each variant
all_metrics = {}

for name in VARIANTS:
    all_metrics[name] = compute_all_metrics(
        name,
        VARIANTS[name],
        intermaps[name],
        sticker_masks[name]
    )

# Display as table
print("\nComprehensive Metrics Table:")
print("=" * 90)

In [ ]:
# Create metrics summary
metrics_table = []
for name, m in all_metrics.items():
    row = {
        'Variant': name,
        'Length': m.length,
        'n_Y': m.n_tyrosine,
        'f_aromatic': f"{m.aromatic_fraction:.3f}",
        'f_sticker': f"{m.sticker_fraction:.3f}",
        'Mean_L': f"{m.mean_linker_length:.1f}",
        'ΔG_proxy': f"{m.delta_G_proxy:.4f}",
        'E_sticker': f"{m.sticker_energy:.4f}",
    }
    metrics_table.append(row)
    
# Print formatted table
headers = list(metrics_table[0].keys())
print(" | ".join(f"{h:>12s}" for h in headers))
print("-" * (14 * len(headers)))
for row in metrics_table:
    print(" | ".join(f"{str(v):>12s}" for v in row.values()))

In [ ]:
# Multi-panel metrics plot
metrics_to_plot = {
    'Sticker Fraction': {name: m.sticker_fraction for name, m in all_metrics.items()},
    'Mean Linker Length': {name: m.mean_linker_length for name, m in all_metrics.items()},
    'ΔG Proxy': {name: m.delta_G_proxy for name, m in all_metrics.items()},
    'Sticker-Sticker Energy': {name: m.sticker_energy for name, m in all_metrics.items()},
    'Aromatic Fraction': {name: m.aromatic_fraction for name, m in all_metrics.items()},
    'Total Energy': {name: m.total_energy for name, m in all_metrics.items()},
}

fig = plot_metrics_multi_panel(
    metrics_to_plot,
    metric_names=list(metrics_to_plot.keys()),
    ncols=3,
    figsize=(14, 8)
)
plt.suptitle('Comprehensive Metrics Comparison', fontsize=14, y=1.02)
plt.show()

In [ ]:
# Correlation: Sticker fraction vs ΔG proxy
fig, ax = plt.subplots(figsize=(8, 6))

f_sticker = [m.sticker_fraction for m in all_metrics.values()]
delta_G = [m.delta_G_proxy for m in all_metrics.values()]
names = list(all_metrics.keys())

ax.scatter(f_sticker, delta_G, c='#3498DB', s=100, edgecolor='black', zorder=5)

# Label points
for i, name in enumerate(names):
    ax.annotate(name, (f_sticker[i], delta_G[i]), xytext=(5, 5), 
                textcoords='offset points', fontsize=10)

ax.set_xlabel('Sticker Fraction')
ax.set_ylabel('ΔG Proxy (kT)')
ax.set_title('Correlation: Sticker Fraction vs Phase Separation Driving Force')

# Fit line
z = np.polyfit(f_sticker, delta_G, 1)
p = np.poly1d(z)
x_line = np.linspace(min(f_sticker), max(f_sticker), 100)
ax.plot(x_line, p(x_line), 'r--', alpha=0.5, label=f'Linear fit')

ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Save metrics to file
import json

output_path = Path("../data/outputs/metrics.json")
metrics_dict = {name: m.to_dict() for name, m in all_metrics.items()}

with open(output_path, 'w') as f:
    json.dump(metrics_dict, f, indent=2)

print(f"Metrics saved to {output_path}")

## Summary

In this notebook we computed:
1. **Sticker-linker metrics**: sticker fraction, linker lengths, κ
2. **Energy metrics**: total energy, mean energy, ΔG proxy, sticker-sticker energy
3. **Sequence metrics**: FCR, NCPR, SCD, κ_charge, aromatic fraction
4. **Spacing metrics**: Ω (omega), clustering
5. **Comprehensive metrics table** combining all measurements

**Key findings**:
- WT has highest sticker fraction (~14%) and strongest ΔG proxy
- AllY_to_S shows dramatic reduction in sticker-based interactions
- AllY_to_F maintains aromatic interactions despite Y→F substitution
- Sticker fraction correlates with ΔG proxy

**Next**: Notebook 06 - Minimal Model and Figures